In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd

from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    precision_recall_curve,
)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline

# Encontrar a pasta ML
candidatos = [
    Path.cwd(),
    Path.cwd() / "ML",
    Path.cwd().parent,
]

BASE = next(
    (
        pasta for pasta in candidatos
        if (pasta / "dados/modelagem/treino_2022_2024.csv").exists()
    ),
    None,
)

if BASE is None:
    raise FileNotFoundError("Pasta de dados do AlertaSP não encontrada.")

DADOS = BASE / "dados/modelagem"
RESULTADOS = BASE / "resultados"
RESULTADOS.mkdir(exist_ok=True)

ALVO = "alvo_registro_amanha"

treino = pd.read_csv(
    DADOS / "treino_2022_2024.csv",
    parse_dates=["data"],
).sort_values("data")

validacao = pd.read_csv(
    DADOS / "validacao_2025.csv",
    parse_dates=["data"],
).sort_values("data")

FEATURES = [
    coluna for coluna in treino.columns
    if coluna not in ("data", ALVO)
]

X_treino = treino[FEATURES]
y_treino = treino[ALVO].astype(int)

X_validacao = validacao[FEATURES]
y_validacao = validacao[ALVO].astype(int)

assert len(FEATURES) == 12
assert treino["data"].max() < validacao["data"].min()

print("Treinamento:", len(treino))
print("Validação:", len(validacao))
print("Features:", len(FEATURES))

display(treino.head())